## Indexing

### Imports

In [1]:
import pandas as pd
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
import os
import pickle


C:\Users\hp\AppData\Roaming\Python\Python314\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


#### CONFIG - Optimized for 8GB RAM laptop

In [2]:
DATA_PATH = "../data/collected.csv"
PERSIST_DIR = "../chroma_db"
BATCH_SIZE = 200          # Smaller batch size to avoid OOM on 8GB RAM
PROGRESS_FILE = "progress.pkl"

### Load the dataset

In [3]:
print("Loading dataset...")
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} fatwas.")

Loading dataset...
Loaded 51470 fatwas.


### Transform DataFrame to documents


In [4]:
documents = []
for _, row in df.iterrows():
    text = f"Question: {row.get('question', '')}\nAnswer: {row.get('answer', '')}"
    metadata = {
        "source": row.get("source", ""),
        "link": row.get("link", "")
    }
    documents.append({"page_content": text, "metadata": metadata})

### Initialize embeddings


In [5]:
print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",                    # Good balance: strong Arabic support + reasonable size
    model_kwargs={'device': 'cpu'},              
    encode_kwargs={'normalize_embeddings': True, 'batch_size': 32}
)


Loading embedding model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

### Initialize Chroma vectorstore


In [6]:
if not os.path.exists(PERSIST_DIR):
    print("Creating new Chroma database...")
    vectorstore = Chroma(
        persist_directory=PERSIST_DIR,
        embedding_function=embeddings
    )
else:
    print("Loading existing Chroma database...")
    vectorstore = Chroma(
        persist_directory=PERSIST_DIR,
        embedding_function=embeddings
    )


Loading existing Chroma database...


### Handeling Errors

#### Resume from last batch if needed


In [7]:
start_idx = 0
if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE, "rb") as f:
        start_idx = pickle.load(f)
    print(f"Resuming from document index {start_idx}...")

Resuming from document index 3900...


### Batch processing


In [ ]:
for i in tqdm(range(start_idx, len(documents), BATCH_SIZE), desc="Indexing batches"):
    batch_docs = documents[i:i + BATCH_SIZE]
    texts = [doc["page_content"] for doc in batch_docs]
    metadatas = [doc["metadata"] for doc in batch_docs]
    
    try:
        vectorstore.add_texts(
            texts=texts,
            metadatas=metadatas
        )
        
        # Save progress
        current_progress = min(i + BATCH_SIZE, len(documents))
        with open(PROGRESS_FILE, "wb") as f:
            pickle.dump(current_progress, f)
            
    except Exception as e:
        print(f"\nError at batch {i}: {e}")
        print("Progress saved. You can resume later.")
        break

Indexing batches:   0%|          | 0/238 [32:21<?, ?it/s]


### Cleanup


In [ ]:
if os.path.exists(PROGRESS_FILE):
    os.remove(PROGRESS_FILE)

print("\n✅ Indexing completed! chroma_db folder is ready.")
print(f"Total documents processed: {len(documents)}")